<a href="https://colab.research.google.com/github/Lordyblade/IbrahimMousaDhani_2411532010_ML2526/blob/main/Praktikum%203/LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import library yang diperlukan.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

Load dan tampilkan Dataset

In [ ]:
# load dataset
dataset_url = 'https://raw.githubusercontent.com/dif-unand/ML_Genap_2526/refs/heads/main/Praktikum3/diabetes.csv'
df = pd.read_csv(dataset_url)
df.head()

Visualisasikan data tersebut untuk melihat karakteristik dari data dan melakukan analisis.
Tambilkan distribusi kolom atau fitur dengan menggunakan histogram


In [ ]:
import math

# Distribution graphs (histogram/bar graph) of column data
def plotPerColumnDistribution(df, nGraphShown, nGraphPerRow):
    nunique = df.nunique()
    df = df[[col for col in df if nunique[col] > 1 and nunique[col] < 50]] # For displaying purposes, pick columns
    nRow, nCol = df.shape
    columnNames = list(df)
    nGraphRow = math.ceil(nCol/nGraphPerRow)
    plt.figure(num = None, figsize = (6 * nGraphPerRow, 8 * nGraphRow), dpi=80, facecolor = 'w', edgecolor ='k')

    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i+1)
        columnDf = df.iloc[:, i]
        if (not np.issubdtype(type(columnDf.iloc[0]), np.number)):
            valueCounts = columnDf.value_counts()
            valueCounts.plot.bar()
        else:
            columnDf.hist()
        plt.ylabel('counts')
        plt.xticks(rotation = 90)
        plt.title(f'{columnNames[i]} (column {i})')
    plt.tight_layout(pad=1.0, w_pad=1.0, h_pad=1.0)
    plt.show()

plotPerColumnDistribution(df, 10, 5)

Tampilkan matrik korelasi untuk setiap kolom/fitur

In [ ]:
#Correlation matrix
def plotCorrelationMatrix(df, graphWidth):
    filename = "diabetes.csv"
    df = df.dropna(axis='columns') # drop columns with NaN
    df = df[[col for col in df if df[col].nunique() > 1]] # keep columns where there are more than 1 unique values
    if df.shape[1] < 2:
        print(f'No correlation plots shown: The number of non-NaN or constant columns ({df.shape[1]}) is less than 2')
        return
    corr = df.corr()
    fig, ax = plt.subplots(num=None, figsize=(graphWidth, graphWidth), dpi=80, facecolor='w', edgecolor='k')
    corrMat = ax.matshow(corr)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.gca().xaxis.tick_bottom()
    plt.colorbar(corrMat)
    plt.title(f'Correlation Matrix for {filename}', fontsize=15)
    plt.show()

plotCorrelationMatrix(df, 8)

Tampilkan sebaran data menggunakan scatter plot

In [ ]:
# Scatter and density plots
def plotScatterMatrix(df, plotSize, textSize):
    df = df.select_dtypes(include=[np.number]) # keep only numerical columns
    #Remove rows and columns that would lead to df being singular
    df = df.dropna(axis='columns')
    df = df[[col for col in df if df[col].nunique() > 1]] # keep columns where there are more than 1 unique values
    columnNames = list(df)
    if len(columnNames) > 10: # reduce the number of columns for matrix inversion of kernel density plots
        columnNames = columnNames[:10]
    df = df[columnNames]
    ax = pd.plotting.scatter_matrix(df, alpha=0.75, figsize=[plotSize, plotSize], diagonal='kde')
    corrs = df.corr().values
    for i, j in zip(*plt.np.triu_indices_from(ax, k=1)):
        ax[i, j].annotate('Corr. coef = %.3f' % corrs[i, j], (0.8, 0.2), xycoords='axes fraction', ha='center', va='center', size=textSize)
    plt.suptitle('Scatter and Density Plot')
    plt.show()

plotScatterMatrix(df, 20, 10)

Selecting features
Pisahkan kolom yang pada data menjadi dua jenis variabel: variabel dependen (atau variabel target) dan
variabel independen (atau variabel fitur).

In [ ]:
#split dataset in features and target variable
feature_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
X = df[feature_cols] # Features
y = df.Outcome # Target variable

print(X.shape)
print(y.shape)

Splitting data

In [ ]:
# Split X and y into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=16)

Pelatihan Model
• Import the LogisticRegression modul
• Buat objek logistic regression classifier menggunakan fungsi LogisticRegression()
• Latih model pada set data pelatihan menggunakan fungsi `fit()` dan lakukan prediksi pada set data
pengujian menggunakan fungsi `predict()`.

In [ ]:
#instantiate the model (using the default parameters)
logreg = LogisticRegression(random_state=16)

#fit the model with data
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)
y_pred